# Dataset Quality Checks (exploratory — safe to discard)

Deeper QA on `data/merged/` beyond what's in `DataPreparation.ipynb`'s class
distribution and per-class sample preview. This notebook is disposable: if a
check here doesn't earn its keep, delete it, no loss.

Run from the repo root. Some cells (full-dataset image decode, perceptual
hashing) take 30-60s each — that's expected, not a hang.


In [ ]:
from itertools import combinations
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml

REPO_ROOT = Path.cwd()
DATASET_ROOT = REPO_ROOT / "data" / "merged"

CLASS_NAMES = yaml.safe_load(open(DATASET_ROOT / "data.yaml"))["names"]
manifest = pd.read_csv(DATASET_ROOT / "merge_manifest.csv")


def load_annotations():
    """One row per box: split, file, class, and corner coordinates
    (normalized 0-1) — the shared source of truth for every geometry-based
    check below."""
    rows = []
    for split_dir in (DATASET_ROOT / "labels").iterdir():
        split = split_dir.name
        for label_file in split_dir.glob("*.txt"):
            for line in label_file.read_text().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                class_id = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
                rows.append((split, label_file.stem, class_id, xc, yc, w, h))
    df = pd.DataFrame(rows, columns=["split", "file", "class_id", "xc", "yc", "w", "h"])
    df["class_name"] = df["class_id"].map(lambda i: CLASS_NAMES[i])
    df["source"] = df["file"].str.split("__", n=1).str[0]
    df["x1"] = df["xc"] - df["w"] / 2
    df["y1"] = df["yc"] - df["h"] / 2
    df["x2"] = df["xc"] + df["w"] / 2
    df["y2"] = df["yc"] + df["h"] / 2
    return df


def iou(b1, b2):
    ax1, ay1, ax2, ay2 = b1
    bx1, by1, bx2, by2 = b2
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    a1 = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    a2 = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0


ann = load_annotations()
print(f"{len(ann)} annotations, {ann['file'].nunique()} images, {len(CLASS_NAMES)} classes")

## 1. Corrupt / unreadable images

Decodes every image once so a bad file surfaces now, not partway through a training run. Takes ~1 min.

In [ ]:
all_image_paths = list((DATASET_ROOT / "images").rglob("*.*"))
bad_images = [p for p in all_image_paths if cv2.imread(str(p)) is None]
print(f"Scanned {len(all_image_paths)} images, {len(bad_images)} unreadable")
for p in bad_images[:20]:
    print(" ", p.relative_to(DATASET_ROOT))

## 2. Bounding box geometry sanity checks

Flags boxes that are geometrically implausible: coordinates outside the image, zero/negative size, or so tiny they're more likely annotation noise than a real detection.

In [ ]:
out_of_range = (ann["x1"] < -0.001) | (ann["y1"] < -0.001) | (ann["x2"] > 1.001) | (ann["y2"] > 1.001)
degenerate = (ann["w"] <= 0) | (ann["h"] <= 0)
tiny = (ann["w"] * ann["h"]) < 0.0002  # < 0.02% of image area

print(f"out-of-range boxes: {out_of_range.sum()}")
print(f"degenerate (zero/negative w or h): {degenerate.sum()}")
print(f"tiny (<0.02% of image area): {tiny.sum()}")

print("\nWorst offenders (degenerate or out-of-range):")
ann[out_of_range | degenerate][["file", "split", "class_name", "xc", "yc", "w", "h"]].head(20)

## 3. Duplicate / overlapping boxes within the same image

Same-class boxes stacked almost exactly on top of each other in one image — can happen from double-annotation or merge artifacts.

In [ ]:
dup_rows = []
for (file, class_id), g in ann.groupby(["file", "class_id"]):
    if len(g) < 2:
        continue
    boxes = g[["x1", "y1", "x2", "y2"]].values
    for i, j in combinations(range(len(boxes)), 2):
        v = iou(boxes[i], boxes[j])
        if v > 0.9:
            dup_rows.append({"file": file, "class_name": CLASS_NAMES[class_id], "iou": round(v, 3)})

dup_df = pd.DataFrame(dup_rows)
print(f"{len(dup_df)} same-class box pairs with IoU > 0.9 (likely double-annotated)")
dup_df.head(20)

## 4. Contradictory positive/negative label overlap

A `helmet` box and a `no-helmet` box (etc.) heavily overlapping in the same spot is a likely labeling error — they're supposed to be mutually exclusive. Pairs are found automatically from whatever `no-X` / `X` classes exist in the current schema.

In [ ]:
opposite_pairs = []
for i, name in enumerate(CLASS_NAMES):
    if name.startswith("no-"):
        positive_name = name[len("no-"):]
        if positive_name in CLASS_NAMES:
            opposite_pairs.append((CLASS_NAMES.index(positive_name), i))
print("Checking pairs:", [(CLASS_NAMES[p], CLASS_NAMES[n]) for p, n in opposite_pairs])

contradiction_rows = []
for file, g in ann.groupby("file"):
    for pos_id, neg_id in opposite_pairs:
        pos_boxes = g[g["class_id"] == pos_id][["x1", "y1", "x2", "y2"]].values
        neg_boxes = g[g["class_id"] == neg_id][["x1", "y1", "x2", "y2"]].values
        for pb in pos_boxes:
            for nb in neg_boxes:
                v = iou(pb, nb)
                if v > 0.5:
                    contradiction_rows.append(
                        {"file": file, "positive": CLASS_NAMES[pos_id], "negative": CLASS_NAMES[neg_id], "iou": round(v, 3)}
                    )

contradiction_df = pd.DataFrame(contradiction_rows)
print(f"\n{len(contradiction_df)} images where a positive and its negative class overlap heavily (IoU > 0.5)")
contradiction_df

## 5. Per-class box size & aspect-ratio distributions

Checks against domain expectations (e.g. `boots` should skew small/occluded per the implementation plan) and flags classes with suspiciously wide spread.

In [ ]:
ann["area_pct"] = ann["w"] * ann["h"] * 100
ann["aspect_ratio"] = ann["w"] / ann["h"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ann.boxplot(column="area_pct", by="class_name", ax=axes[0], rot=45, showfliers=False)
axes[0].set_title("Box area, % of image (outliers hidden)")
axes[0].set_ylabel("% of image area")
ann.boxplot(column="aspect_ratio", by="class_name", ax=axes[1], rot=45, showfliers=False)
axes[1].set_title("Box aspect ratio, w/h (outliers hidden)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Domain-shift check: brightness/contrast by source

The three sources were shot very differently; this puts a number on it. Relevant since the plan's augmentation strategy (brightness/shadow) is designed around exactly this kind of shift.

In [ ]:
brightness_rows = []
for source, g in manifest.groupby("source"):
    sample = g.sample(min(300, len(g)), random_state=0)
    for _, r in sample.iterrows():
        p = DATASET_ROOT / "images" / r["split"] / r["merged_filename"]
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        brightness_rows.append({"source": source, "brightness": img.mean(), "contrast": img.std()})

bdf = pd.DataFrame(brightness_rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bdf.boxplot(column="brightness", by="source", ax=axes[0])
axes[0].set_title("Brightness by source (sample of 300/source)")
bdf.boxplot(column="contrast", by="source", ax=axes[1])
axes[1].set_title("Contrast by source (sample of 300/source)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 7. Random unfiltered grid overview

`DataPreparation.ipynb` only samples *within* a class. This is a plain random sample across the whole merged set — a faster "does this look like a coherent dataset" gut check before drilling into per-class detail.

In [ ]:
def draw_all_boxes(image_path, label_path):
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, bw, bh = line.split()
        class_id = int(class_id)
        xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
        x1, y1 = int(xc - bw / 2), int(yc - bh / 2)
        x2, y2 = int(xc + bw / 2), int(yc + bh / 2)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, CLASS_NAMES[class_id], (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return img


sample = manifest.sample(12, random_state=1)
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, (_, row) in zip(axes.flatten(), sample.iterrows()):
    image_path = DATASET_ROOT / "images" / row["split"] / row["merged_filename"]
    label_path = DATASET_ROOT / "labels" / row["split"] / f"{Path(row['merged_filename']).stem}.txt"
    ax.imshow(draw_all_boxes(image_path, label_path))
    ax.set_title(f"{row['source']} / {row['split']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Person-count-per-image distribution

Relevant to the plan's own "Helmet Snatching" concern (Hungarian matching in crowded scenes) — how much genuinely crowded data exists to stress-test that later.

In [ ]:
if "person" in CLASS_NAMES:
    person_id = CLASS_NAMES.index("person")
    person_counts = ann[ann["class_id"] == person_id].groupby("file").size()

    plt.figure(figsize=(8, 5))
    person_counts.value_counts().sort_index().plot(kind="bar")
    plt.xlabel("Number of person boxes in image")
    plt.ylabel("Number of images")
    plt.title("Crowd-size distribution (images containing >=1 person)")
    plt.tight_layout()
    plt.show()
    print(f"{len(person_counts)} images contain >=1 person; max in one image: {person_counts.max()}")
else:
    print("no 'person' class in the current schema")

## 9. Cross-split leakage: augmented siblings & near-duplicate images

**This is the one that found a real issue.** Two of the three sources are
Roboflow exports, whose augmentation pipeline creates several versions of
each source photo (rotate/crop/brightness variants) sharing a filename
pattern `<base>_jpg.rf.<hash>.<ext>`. The dataset's stratified split
(`scripts/build_dataset.py`) currently splits **per image**, not per base
photo — so siblings of the same source photo can land in different splits,
letting near-duplicates of a training image leak into val/test and inflate
those metrics. This notebook only *detects and reports* it — the fix (group
the stratification by base-image family) would be a change to
`scripts/build_dataset.py`, done separately if you want it.


In [ ]:
import re


def base_family(fn):
    m = re.match(r"^(.*)_jpg\.rf\.[0-9a-f]+\.\w+$", fn)
    return m.group(1) if m else None


manifest["family"] = manifest["merged_filename"].apply(base_family)
fam = manifest[manifest["family"].notna()]
family_splits = fam.groupby(["source", "family"])["split"].nunique()
leaked = family_splits[family_splits > 1]

print(f"{fam.groupby(['source', 'family']).ngroups} roboflow-style image families found")
print(f"{len(leaked)} of them have members split across more than one of train/val/test")
leaked.sort_values(ascending=False).head(10)

### Near-duplicates without a shared filename pattern

`ketakichalke-boots` doesn't use Roboflow's naming convention, so the check
above can't catch duplicates there (or duplicates *across* sources). This
uses a coarse 64-bit average hash (aHash) instead — cheap enough to run on
the whole dataset, but it **will** produce false-positive collisions
(visually simple/uniform images can hash identically despite different
content). Treat the count below as a shortlist to eyeball, not a definitive
duplicate count — the image grid after it shows a few actual examples so
you can judge for yourself.


In [ ]:
def ahash(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(img, (8, 8), interpolation=cv2.INTER_AREA)
    bits = (small > small.mean()).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


hash_map = {}
for _, row in manifest.iterrows():
    p = DATASET_ROOT / "images" / row["split"] / row["merged_filename"]
    h = ahash(p)
    hash_map.setdefault(h, []).append((row["merged_filename"], row["split"]))

cross_split_groups = [v for v in hash_map.values() if len(v) > 1 and len({s for _, s in v}) > 1]
print(f"{len(hash_map)} distinct hashes across {len(manifest)} images")
print(f"{sum(1 for v in hash_map.values() if len(v) > 1)} hash groups with >1 image")
print(f"{len(cross_split_groups)} of those span more than one split — review these before trusting val/test")

In [ ]:
def show_group(paths_splits, title):
    fig, axes = plt.subplots(1, len(paths_splits), figsize=(5 * len(paths_splits), 5))
    if len(paths_splits) == 1:
        axes = [axes]
    for ax, (fn, split) in zip(axes, paths_splits):
        img = cv2.cvtColor(cv2.imread(str(DATASET_ROOT / "images" / split / fn)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{split}\n{fn}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# First few candidates — inspect visually to judge real duplicate vs. hash
# collision on two simple/different images.
for group in cross_split_groups[:3]:
    show_group(group, "Candidate cross-split near-duplicate (verify visually)")